# Label Swapping
[Learning Imbalanced Data with Beneficial Label Noise](https://openreview.net/forum?id=AZT4EiONRQ)

[Algorithm](algo.png)

In [ ]:
import torch
import torch.nn.functional as F

def apply_lnr(logits, labels_true, epoch_cur, epoch_start, t_flip, classes_maj):

    if epoch_cur < epoch_start:
        return labels_true

    probs = F.softmax(logits, dim=1)

    mu = probs.mean(dim=0)
    std = probs.std(dim=0)
    z_scores = (probs - mu) / (std + 1e-8)

    flip_rate = torch.clamp(torch.tanh(z_scores - t_flip), min=0.0)

    device = labels_true.device
    is_majority = torch.tensor([lbl.item() in classes_maj for lbl in labels_true], device=device)
    
    flip_rate = flip_rate * is_majority.unsqueeze(1) 

    is_flip = torch.bernoulli(flip_rate)

    labels_new = labels_true.clone()
    idxs_flip = torch.nonzero(is_flip, as_tuple=True)
    
    classes_new = torch.argmax(flip_rate, dim=1)

    batch_indices = idxs_flip[0]
    labels_new[batch_indices] = classes_new[batch_indices]

    return labels_new